# Kaggle GPU → OpenAI-compatible LLM API for agentic coding

This notebook turns a free Kaggle GPU session into a public, authenticated,
**OpenAI-compatible** endpoint (`/v1/chat/completions`, `/v1/completions`, `/v1/models`)
that coding agents such as Aider, OpenCode, Cline, Roo Code, Continue or the Codex CLI can use.

**Architecture**

```
agent(s) ──HTTPS──▶ tunnel (ngrok / Cloudflare) ──▶ :8000 async gateway (auth, least-busy routing, streaming)
                                                         ├──▶ :8001 vLLM (GPU 0 / or GPU 0+1 tensor-parallel)
                                                         └──▶ :8002 vLLM (GPU 1)            [replica mode]
```

**How it handles many connections**

* **vLLM continuous batching + PagedAttention**: dozens of requests share each forward pass instead of queueing.
* **Prefix caching**: agents resend the same long system prompt / repo context every turn; the shared prefix is computed once.
* **Chunked prefill**: a huge prompt from one client doesn't freeze token streaming for everyone else.
* **Replica mode**: on T4×2 with a model that fits on one GPU, two independent engines double throughput; the gateway sends each request to the least-busy one and retries on another if one is down.
* **Async gateway**: one aiohttp process with unlimited keep-alive connections, true streaming passthrough (SSE), and API-key auth.

**Before you run**

1. *Settings → Accelerator*: **GPU T4 ×2** (recommended) or P100.
2. *Settings → Internet*: **On** (requires a phone-verified Kaggle account).
3. *Add-ons → Secrets* (all optional):
   * `API_KEY`: the key clients must send. If missing, a random one is generated and printed.
   * `NGROK_TOKEN`: ngrok authtoken (recommended tunnel; supports streaming).
   * `CF_TUNNEL_TOKEN`: token of a named Cloudflare tunnel whose public hostname points to `http://localhost:8000`.
   * `HF_TOKEN`: only needed for gated models.
4. Run all cells. The last cell keeps the session alive and prints live load stats.
   For unattended use, choose **Save Version → Save & Run All (Commit)**: the session then runs up to 12 h without a browser tab open.

> Kaggle gives ~30 GPU-hours/week and a 12 h session limit. Read Kaggle's terms for your use case, and keep the API key secret: anyone with the URL and key can use your GPU.

## 1. Configuration

Pick a model preset or set your own. All presets are 4-bit AWQ in **fp16**, because T4/P100 GPUs have no bf16 support.

| Preset | VRAM (weights) | Best mode on T4×2 | Notes |
|---|---|---|---|
| `qwen2.5-coder-14b` | ~10 GB | `tp` | Default. Strong coder with reliable tool calls. |
| `qwen2.5-coder-7b`  | ~5.5 GB | `replicas` | Highest concurrency: 2 engines, one per GPU. |
| `qwen2.5-coder-32b` | ~19 GB | `tp` | Smartest choice that fits; short context, few concurrent users. |
| `qwen3-8b`          | ~6 GB | `replicas` | General + coding, hybrid thinking. |

In [ ]:
# ============================ USER SETTINGS ============================
PRESET = "qwen2.5-coder-14b"      # one of PRESETS below, or "custom"

# Only used when PRESET == "custom"
CUSTOM = dict(
    model="Qwen/Qwen2.5-Coder-14B-Instruct-AWQ",
    tool_parser="hermes",         # hermes | qwen3_coder | llama3_json | mistral | ...
    max_model_len=32768,
    mode="tp",                    # "tp" = one engine across all GPUs, "replicas" = one engine per GPU
    extra_args=[],
)

SERVED_MODEL_NAME = "kaggle-coder"   # name clients put in "model"
MAX_NUM_SEQS      = 48               # max concurrent sequences per engine (raise for more users, lower if OOM)
GPU_MEM_UTIL      = 0.92             # fraction of each GPU vLLM may use
TUNNEL            = "auto"           # auto | ngrok | cloudflare | cloudflare-quick | none
GATEWAY_PORT      = 8000
VLLM_VERSION      = ""               # "" = latest; pin e.g. "0.10.1.1" if a new release breaks on T4
EXTRA_VLLM_ARGS   = []               # e.g. ["--enforce-eager"] to save VRAM (slower)
# =======================================================================

PRESETS = {
    "qwen2.5-coder-14b": dict(model="Qwen/Qwen2.5-Coder-14B-Instruct-AWQ", tool_parser="hermes",
                              max_model_len=32768, mode="tp", extra_args=[]),
    "qwen2.5-coder-7b":  dict(model="Qwen/Qwen2.5-Coder-7B-Instruct-AWQ", tool_parser="hermes",
                              max_model_len=32768, mode="replicas", extra_args=[]),
    "qwen2.5-coder-32b": dict(model="Qwen/Qwen2.5-Coder-32B-Instruct-AWQ", tool_parser="hermes",
                              max_model_len=16384, mode="tp", extra_args=[]),
    "qwen3-8b":          dict(model="Qwen/Qwen3-8B-AWQ", tool_parser="hermes",
                              max_model_len=32768, mode="replicas",
                              extra_args=["--reasoning-parser", "qwen3"]),
}
CFG = CUSTOM if PRESET == "custom" else PRESETS[PRESET]
print(CFG)

## 2. Secrets and GPU check

In [ ]:
import os, secrets, subprocess

def kaggle_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

API_KEY         = kaggle_secret("API_KEY") or ("sk-kaggle-" + secrets.token_urlsafe(24))
NGROK_TOKEN     = kaggle_secret("NGROK_TOKEN")
CF_TUNNEL_TOKEN = kaggle_secret("CF_TUNNEL_TOKEN")
HF_TOKEN        = kaggle_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout)
import torch
NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS > 0, "No GPU found: set Settings → Accelerator to GPU T4 x2"
print("GPUs:", NUM_GPUS)
print("Secrets found:", {k: bool(v) for k, v in
      dict(API_KEY=kaggle_secret("API_KEY"), NGROK_TOKEN=NGROK_TOKEN,
           CF_TUNNEL_TOKEN=CF_TUNNEL_TOKEN, HF_TOKEN=HF_TOKEN).items()})

## 3. Install vLLM, the gateway dependencies and cloudflared (~3-5 min)

In [ ]:
spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
!pip install -q -U "{spec}" aiohttp pyngrok openai 2>&1 | grep -vE "dependency resolver|incompatible|^\s*$" | tail -n 5
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!python -c "import vllm; print('vLLM', vllm.__version__)"
!cloudflared --version

## 4. Gateway (auth + least-busy load balancing + streaming)

A small aiohttp reverse proxy. It
* checks `Authorization: Bearer <key>` (or `x-api-key`),
* sends each request to the backend with the fewest in-flight requests,
* retries on another backend if one refuses the connection,
* streams responses chunk by chunk, so SSE token streaming works,
* serves an unauthenticated `GET /health` for monitoring.

In [ ]:
GATEWAY_SRC = r"""
import asyncio, os, sys, time
from aiohttp import web, ClientSession, ClientTimeout, TCPConnector, ClientConnectorError

API_KEY  = os.environ["GW_API_KEY"]
BACKENDS = os.environ["GW_BACKENDS"].split(",")
PORT     = int(os.environ.get("GW_PORT", "8000"))
inflight = {b: 0 for b in BACKENDS}
stats    = {"total": 0, "errors": 0, "started": time.time()}
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te",
       "trailers", "transfer-encoding", "upgrade", "host", "content-length",
       "authorization", "x-api-key"}

def authorized(req):
    auth = req.headers.get("Authorization", "")
    key = auth[7:] if auth.lower().startswith("bearer ") else req.headers.get("x-api-key", "")
    return key == API_KEY

async def health(req):
    return web.json_response({"ok": True, "inflight": inflight, **stats})

async def proxy(req):
    if req.method == "OPTIONS":
        return web.Response(headers={"Access-Control-Allow-Origin": "*",
            "Access-Control-Allow-Headers": "*", "Access-Control-Allow-Methods": "*"})
    if not authorized(req):
        return web.json_response({"error": {"message": "Invalid API key", "type": "auth_error"}}, status=401)
    stats["total"] += 1
    body = await req.read()
    headers = {k: v for k, v in req.headers.items() if k.lower() not in HOP}
    tried = set()
    while True:
        candidates = [b for b in BACKENDS if b not in tried]
        if not candidates:
            stats["errors"] += 1
            return web.json_response({"error": {"message": "No backend available"}}, status=503)
        backend = min(candidates, key=lambda b: inflight[b])
        tried.add(backend)
        inflight[backend] += 1
        resp = None
        try:
            async with req.app["session"].request(req.method, backend + req.rel_url.path_qs,
                                                  data=body, headers=headers) as up:
                resp = web.StreamResponse(status=up.status, headers={
                    k: v for k, v in up.headers.items() if k.lower() not in HOP})
                resp.headers["Access-Control-Allow-Origin"] = "*"
                resp.headers["X-Accel-Buffering"] = "no"
                await resp.prepare(req)
                async for chunk in up.content.iter_any():
                    await resp.write(chunk)
                await resp.write_eof()
                return resp
        except ClientConnectorError:
            if resp is None:
                continue          # backend down before anything was sent: try the next one
            raise
        except (ConnectionResetError, asyncio.CancelledError):
            raise                 # client went away
        finally:
            inflight[backend] -= 1

async def on_startup(app):
    app["session"] = ClientSession(connector=TCPConnector(limit=0, keepalive_timeout=60),
                                   timeout=ClientTimeout(total=None, sock_connect=5, sock_read=1800),
                                   auto_decompress=False)

async def on_cleanup(app):
    await app["session"].close()

app = web.Application(client_max_size=256 * 1024 ** 2)
app.on_startup.append(on_startup)
app.on_cleanup.append(on_cleanup)
app.router.add_get("/health", health)
app.router.add_route("*", "/{tail:.*}", proxy)
web.run_app(app, host="0.0.0.0", port=PORT, backlog=4096, access_log=None)
"""
with open("/kaggle/working/gateway.py" if os.path.isdir("/kaggle/working") else "gateway.py", "w") as f:
    f.write(GATEWAY_SRC)
GATEWAY_PATH = os.path.abspath(f.name)
print("Wrote", GATEWAY_PATH)

## 5. Launch vLLM engine(s)

`tp` mode runs one engine split across all GPUs (bigger models, longer context).
`replicas` mode runs one engine per GPU (more total throughput for models that fit on one GPU).
The first launch downloads the weights, so it can take 5-10 minutes.

In [ ]:
import time, requests, signal

LOG_DIR = "/kaggle/working/logs" if os.path.isdir("/kaggle/working") else "logs"
os.makedirs(LOG_DIR, exist_ok=True)

mode = CFG["mode"] if NUM_GPUS > 1 else "tp"
if mode == "tp":
    engines = [dict(gpus=",".join(map(str, range(NUM_GPUS))), tp=NUM_GPUS, port=8001)]
else:
    engines = [dict(gpus=str(i), tp=1, port=8001 + i) for i in range(NUM_GPUS)]

def vllm_cmd(e):
    return [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", CFG["model"],
        "--served-model-name", SERVED_MODEL_NAME, CFG["model"],
        "--host", "127.0.0.1", "--port", str(e["port"]),
        "--tensor-parallel-size", str(e["tp"]),
        "--dtype", "half",                       # T4/P100: no bf16
        "--max-model-len", str(CFG["max_model_len"]),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--enable-prefix-caching",               # agents resend the same long prefix every turn
        "--enable-chunked-prefill",              # long prompts don't stall other streams
        "--enable-auto-tool-choice", "--tool-call-parser", CFG["tool_parser"],
        "--disable-log-requests",
        "--trust-remote-code",
        *CFG.get("extra_args", []), *EXTRA_VLLM_ARGS,
    ]

def start_engine(e):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=e["gpus"], VLLM_WORKER_MULTIPROC_METHOD="spawn")
    log = open(f"{LOG_DIR}/vllm_{e['port']}.log", "a")
    e["proc"] = subprocess.Popen(vllm_cmd(e), env=env, stdout=log, stderr=subprocess.STDOUT,
                                 start_new_session=True)
    print(f"Started vLLM on GPU(s) {e['gpus']} → :{e['port']} (pid {e['proc'].pid})")

def wait_ready(e, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if e["proc"].poll() is not None:
            print(open(f"{LOG_DIR}/vllm_{e['port']}.log").read()[-4000:])
            raise RuntimeError(f"vLLM :{e['port']} exited; see log above")
        try:
            if requests.get(f"http://127.0.0.1:{e['port']}/health", timeout=2).ok:
                print(f":{e['port']} ready after {time.time()-t0:.0f}s")
                return
        except requests.RequestException:
            pass
        time.sleep(5)
    raise TimeoutError(f"vLLM :{e['port']} not ready")

def pkill(*patterns):
    for p in patterns:
        subprocess.run(["pkill", "-f", p])

# Stop anything left over from a previous run of this cell
pkill("vllm.entrypoints.openai.api_server", "gateway.py"); time.sleep(3)

print("Mode:", mode, "| model:", CFG["model"])
print(" ".join(vllm_cmd(engines[0])))
for i, e in enumerate(engines):
    start_engine(e)
    if i == 0 and len(engines) > 1:
        wait_ready(e)   # first engine downloads the weights; the others reuse the HF cache
for e in engines:
    wait_ready(e)
!tail -n 5 {LOG_DIR}/vllm_8001.log

## 6. Start the gateway

In [ ]:
gw_env = dict(os.environ, GW_API_KEY=API_KEY, GW_PORT=str(GATEWAY_PORT),
              GW_BACKENDS=",".join(f"http://127.0.0.1:{e['port']}" for e in engines))
gw_log = open(f"{LOG_DIR}/gateway.log", "a")
gateway_proc = subprocess.Popen(["python", GATEWAY_PATH], env=gw_env, stdout=gw_log,
                                stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(30):
    try:
        if requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
print(requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/health").json())
r = requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
print(r.status_code, [m["id"] for m in r.json()["data"]])

## 7. Expose it publicly

`TUNNEL="auto"` picks the first available of:

1. **ngrok** (`NGROK_TOKEN` secret). Free account; supports streaming; you can use your free static domain.
2. **Cloudflare named tunnel** (`CF_TUNNEL_TOKEN` secret). Stable hostname on your own domain; best for many users.
3. **Cloudflare quick tunnel**. No account needed, random `*.trycloudflare.com` URL. Quick tunnels are meant for testing:
   Cloudflare limits concurrent requests and may not stream Server-Sent Events, so tokens can arrive all at once.

In [ ]:
import re
PUBLIC_URL = None
tunnel_proc = None

def start_ngrok():
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    domain = kaggle_secret("NGROK_DOMAIN")          # optional: your free static domain
    t = ngrok.connect(GATEWAY_PORT, "http", **({"domain": domain} if domain else {}))
    return t.public_url

def start_cloudflare_named():
    global tunnel_proc
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "run",
                                    "--token", CF_TUNNEL_TOKEN],
                                   stdout=open(f"{LOG_DIR}/cloudflared.log", "a"),
                                   stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(8)
    return kaggle_secret("CF_HOSTNAME") or "(your Cloudflare tunnel hostname)"

def start_cloudflare_quick():
    global tunnel_proc
    logf = f"{LOG_DIR}/cloudflared_quick.log"
    open(logf, "w").close()
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate",
                                    "--url", f"http://127.0.0.1:{GATEWAY_PORT}"],
                                   stdout=open(logf, "a"), stderr=subprocess.STDOUT,
                                   start_new_session=True)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(logf).read())
        if m:
            return m.group(0)
        time.sleep(1)
    raise RuntimeError("Quick tunnel URL not found; see " + logf)

pkill("cloudflared")
choice = TUNNEL
if choice == "auto":
    choice = "ngrok" if NGROK_TOKEN else "cloudflare" if CF_TUNNEL_TOKEN else "cloudflare-quick"
if choice == "ngrok":
    PUBLIC_URL = start_ngrok()
elif choice == "cloudflare":
    PUBLIC_URL = start_cloudflare_named()
elif choice == "cloudflare-quick":
    PUBLIC_URL = start_cloudflare_quick()
else:
    PUBLIC_URL = f"http://127.0.0.1:{GATEWAY_PORT}"
if not PUBLIC_URL.startswith("http"):
    PUBLIC_URL = "https://" + PUBLIC_URL
BASE_URL = PUBLIC_URL.rstrip("/") + "/v1"

print(f"""
Tunnel:   {choice}
Base URL: {BASE_URL}
API key:  {API_KEY}
Model:    {SERVED_MODEL_NAME}
""")

## 8. Smoke tests: chat, tool calling, concurrency

In [ ]:
from openai import OpenAI
time.sleep(3)
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=600)

r = client.chat.completions.create(model=SERVED_MODEL_NAME, max_tokens=200, messages=[
    {"role": "user", "content": "Write a Python one-liner that reverses the words in a string."}])
print(r.choices[0].message.content)

tools = [{"type": "function", "function": {
    "name": "read_file", "description": "Read a file from the repository",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}}]
r = client.chat.completions.create(model=SERVED_MODEL_NAME, tools=tools, tool_choice="auto", messages=[
    {"role": "user", "content": "Open src/main.py so you can look at it."}])
print("tool calls:", r.choices[0].message.tool_calls)

print("streaming: ", end="")
for chunk in client.chat.completions.create(model=SERVED_MODEL_NAME, stream=True, max_tokens=40,
        messages=[{"role": "user", "content": "Count from 1 to 15."}]):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

In [ ]:
# Concurrency benchmark: N simultaneous agents through the public URL
import asyncio
from openai import AsyncOpenAI

N_CLIENTS, MAX_TOKENS = 32, 256

async def one(aclient, i):
    t0 = time.time()
    r = await aclient.chat.completions.create(model=SERVED_MODEL_NAME, max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": f"Task #{i}: write a Python function implementing binary search, with docstring."}])
    return r.usage.completion_tokens, time.time() - t0

async def bench():
    aclient = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=900)
    t0 = time.time()
    res = await asyncio.gather(*(one(aclient, i) for i in range(N_CLIENTS)), return_exceptions=True)
    wall = time.time() - t0
    ok = [x for x in res if not isinstance(x, Exception)]
    toks = sum(t for t, _ in ok)
    print(f"{len(ok)}/{N_CLIENTS} succeeded in {wall:.1f}s | {toks/wall:.0f} tok/s aggregate | "
          f"avg latency {sum(l for _, l in ok)/max(len(ok),1):.1f}s")
    for x in res:
        if isinstance(x, Exception):
            print("error:", repr(x)[:200]); break

await bench()

## 9. Connect your coding agent

Every tool below speaks the OpenAI API. Use the **Base URL**, **API key** and **model** printed above.
Run the next cell for copy-paste snippets with your values filled in.

In [ ]:
print(f"""# ---- Generic (most tools read these) ----
export OPENAI_API_BASE="{BASE_URL}"
export OPENAI_BASE_URL="{BASE_URL}"
export OPENAI_API_KEY="{API_KEY}"

# ---- Aider ----
aider --model openai/{SERVED_MODEL_NAME}

# ---- OpenCode: ~/.config/opencode/opencode.json ----
{{
  "$schema": "https://opencode.ai/config.json",
  "provider": {{
    "kaggle": {{
      "npm": "@ai-sdk/openai-compatible",
      "name": "Kaggle GPU",
      "options": {{ "baseURL": "{BASE_URL}", "apiKey": "{API_KEY}" }},
      "models": {{ "{SERVED_MODEL_NAME}": {{ "name": "{CFG['model']}" }} }}
    }}
  }}
}}

# ---- Codex CLI: ~/.codex/config.toml ----
model = "{SERVED_MODEL_NAME}"
model_provider = "kaggle"
[model_providers.kaggle]
name = "Kaggle GPU"
base_url = "{BASE_URL}"
env_key = "OPENAI_API_KEY"
wire_api = "chat"

# ---- Cline / Roo Code / Continue / Kilo ----
Provider: "OpenAI Compatible" | Base URL: {BASE_URL} | API key: {API_KEY} | Model: {SERVED_MODEL_NAME}
Context window: {CFG['max_model_len']}
""")

## 10. Keep alive + monitor (+ auto-restart crashed engines)

This cell blocks. It prints running/waiting requests and KV-cache usage every minute and restarts any engine that dies.
Interrupt it to get the notebook back; the servers keep running in the background.

In [ ]:
def metrics(port):
    try:
        txt = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=3).text
    except requests.RequestException:
        return "down"
    def g(name):
        vals = [float(l.split()[-1]) for l in txt.splitlines()
                if l.startswith(name) and not l.startswith("#")]
        return sum(vals) if vals else float("nan")
    kv = g("vllm:gpu_cache_usage_perc")
    if kv != kv:
        kv = g("vllm:kv_cache_usage_perc")
    return f"run={g('vllm:num_requests_running'):.0f} wait={g('vllm:num_requests_waiting'):.0f} kv={kv*100:.0f}%"

print(f"Serving {BASE_URL}  (model '{SERVED_MODEL_NAME}')")
try:
    while True:
        for e in engines:
            if e["proc"].poll() is not None:
                print(time.strftime("%H:%M:%S"), f"engine :{e['port']} died, restarting")
                start_engine(e); wait_ready(e)
        if gateway_proc.poll() is not None:
            print("gateway died, restarting")
            gateway_proc = subprocess.Popen(["python", GATEWAY_PATH], env=gw_env, stdout=gw_log,
                                            stderr=subprocess.STDOUT, start_new_session=True)
        if tunnel_proc is not None and tunnel_proc.poll() is not None:
            print("tunnel died; re-run section 7 (a quick tunnel gets a NEW URL)")
        gw = requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=3).json()
        print(time.strftime("%H:%M:%S"), f"total={gw['total']} errors={gw['errors']} |",
              " | ".join(f":{e['port']} {metrics(e['port'])}" for e in engines), flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("Monitor stopped (servers still running).")

## Troubleshooting

* **CUDA out of memory at startup**: lower `MAX_NUM_SEQS` (e.g. 16), `max_model_len`, or `GPU_MEM_UTIL`, or add `--enforce-eager` to `EXTRA_VLLM_ARGS`.
* **Attention-backend / "compute capability" errors on T4**: pin an older vLLM with `VLLM_VERSION = "0.10.1.1"`, or set `os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"` (or `"FLEX_ATTENTION"`) before section 5.
* **Tool calls come back as plain text**: the `tool_parser` must match the model's chat template (`hermes` for Qwen2.5/Qwen3, `qwen3_coder` for Qwen3-Coder, `llama3_json` for Llama 3.x, `mistral` for Mistral).
* **Requests queue (`wait` > 0 for long)**: the GPU is saturated. Use a smaller model in `replicas` mode, or lower `max_tokens` on the client.
* **Responses don't stream through the quick tunnel**: use ngrok or a named Cloudflare tunnel.
* **Logs**: `/kaggle/working/logs/` (`vllm_8001.log`, `gateway.log`, `cloudflared*.log`).

In [ ]:
# Shut everything down (frees the GPU without restarting the kernel)
pkill("vllm.entrypoints.openai.api_server", "gateway.py", "cloudflared", "ngrok")